In [1]:
import pandas as pd
path = r"C:\Users\ashle\OneDrive\Desktop\UManitoba\Courses\FALL 2026\COMP 3380\GroupProject\DataCleaning\crew.csv"

In [2]:
bad_lines = []

def catch_bad_line(line):
    bad_lines.append(line)
    return None

df = pd.read_csv(path, engine='python', on_bad_lines=catch_bad_line)

In [3]:
print(f"Loaded {len(df)} valid rows.")
print(f"Captured {len(bad_lines)} malformed rows for cleaning.")
# Print out the data
df

Loaded 63632 valid rows.
Captured 0 malformed rows for cleaning.


,id,movie_id,person_id,name,job,department,movie_title,movie_release_date
0,29930,2,16767,Aki Kaurismäki,Director,Directing,Ariel,1988-10-21
1,29932,2,16767,Aki Kaurismäki,Producer,Production,Ariel,1988-10-21
2,29931,2,16767,Aki Kaurismäki,Writer,Writing,Ariel,1988-10-21
3,1441,11,1,George Lucas,Director,Directing,Star Wars,1977-05-25
4,1438,11,1,George Lucas,Executive Producer,Production,Star Wars,1977-05-25
...,...,...,...,...,...,...,...,...
63627,3283,1620991,2075953,Ian Rumsey,Executive Producer,Production,The Investigation of Lucy Letby,2026-02-04
63628,3284,1620991,3859938,Caroline Short,Executive Producer,Production,The Investigation of Lucy Letby,2026-02-04
63629,46018,1627792,4866830,Sid T. Pascua,Director,Directing,Sagaran,2026-02-06
63630,21562,1627796,3555327,King Abalos,Director,Directing,Pansamantala,2026-02-13


In [4]:
# Data overview
df.info()
print("Data types seems to be consistent and there are no null values.")

print("Duplicate rows count:", df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 63632 entries, 0 to 63631
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   id                  63632 non-null  int64
 1   movie_id            63632 non-null  int64
 2   person_id           63632 non-null  int64
 3   name                63632 non-null  str  
 4   job                 63632 non-null  str  
 5   department          63632 non-null  str  
 6   movie_title         63632 non-null  str  
 7   movie_release_date  63302 non-null  str  
dtypes: int64(3), str(5)
memory usage: 3.9 MB
Data types seems to be consistent and there are no null values.
Duplicate rows count: 0


In [5]:
# Data cleaning start
# 1. TODO: NaN values

print("Sum of NaN values:")
sumOfNaNValues = df.isna().sum()
sumOfNaNValues
print("There are no NaN values except for movie_release_date column (which we will drop since we don't need it in this csv")

Sum of NaN values:
There are no NaN values except for movie_release_date column (which we will drop since we don't need it in this csv


In [6]:
# 2. Drop columns that we don't need from ER diagram

# In the ER diagram, cast (Person) entity only has 4 attributes: personID, personName, castOrder, and character.
# Since we'll need a connection to movie, movie_id would be enough
df = df.drop(columns=['id','movie_title','movie_release_date'])
df

,movie_id,person_id,name,job,department
0,2,16767,Aki Kaurismäki,Director,Directing
1,2,16767,Aki Kaurismäki,Producer,Production
2,2,16767,Aki Kaurismäki,Writer,Writing
3,11,1,George Lucas,Director,Directing
4,11,1,George Lucas,Executive Producer,Production
...,...,...,...,...,...
63627,1620991,2075953,Ian Rumsey,Executive Producer,Production
63628,1620991,3859938,Caroline Short,Executive Producer,Production
63629,1627792,4866830,Sid T. Pascua,Director,Directing
63630,1627796,3555327,King Abalos,Director,Directing


In [7]:
# 3. String format: Whitespace cleanup
# Identify rows with two or more consecutive spaces

# Strip leading/trailing spaces and collapse multiple spaces for all text columns
df_inconsistentWhitespace = df[
    df['name'].str.contains(r'\s{2,}', regex=True, na=False) |
    df['job'].str.contains(r'\s{2,}', regex=True, na=False) |
    df['department'].str.contains(r'\s{2,}', regex=True, na=False)
]

df_inconsistentWhitespace[['person_id', 'name', 'job', 'department']]

,person_id,name,job,department
12984,1037124,Edward H. Hamm Jr. \t,Executive Producer,Production
18891,1037124,Edward H. Hamm Jr. \t,Executive Producer,Production
21252,582620,John Constantine,Producer,Production
36714,1037124,Edward H. Hamm Jr. \t,Producer,Production
38562,1037124,Edward H. Hamm Jr. \t,Executive Producer,Production
40191,1037124,Edward H. Hamm Jr. \t,Executive Producer,Production
49961,3134299,Liv Krusinski,Executive Producer,Production


In [8]:
# Clean leading/trailing whitespace, tabs, and collapse multiple spaces into one
for col in ['name', 'job', 'department']:
    df[col] = df[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

# Confirm whitespace has been cleaned up
df_inconsistentWhitespace = df[
    df['name'].str.contains(r'\s{2,}|\t', regex=True, na=False) |
    df['job'].str.contains(r'\s{2,}|\t', regex=True, na=False) |
    df['department'].str.contains(r'\s{2,}|\t', regex=True, na=False)
]

df_inconsistentWhitespace[['person_id', 'name', 'job', 'department']]

,person_id,name,job,department


In [9]:
# Print Cleaned up data
df

,movie_id,person_id,name,job,department
0,2,16767,Aki Kaurismäki,Director,Directing
1,2,16767,Aki Kaurismäki,Producer,Production
2,2,16767,Aki Kaurismäki,Writer,Writing
3,11,1,George Lucas,Director,Directing
4,11,1,George Lucas,Executive Producer,Production
...,...,...,...,...,...
63627,1620991,2075953,Ian Rumsey,Executive Producer,Production
63628,1620991,3859938,Caroline Short,Executive Producer,Production
63629,1627792,4866830,Sid T. Pascua,Director,Directing
63630,1627796,3555327,King Abalos,Director,Directing


In [10]:
#4. Final cleanup (renaming to match ER diagram)

df = df.rename(columns={
    'movie_id': 'movieID',
    'person_id': 'personID',
    'name': 'personName'
})

In [11]:
# Save cleaned data in csv
clean_path = r"C:\Users\ashle\OneDrive\Desktop\UManitoba\Courses\FALL 2026\COMP 3380\GroupProject\DataCleaning\CleanDatas\crew_cleaned.csv"
df.to_csv(clean_path, index=False)